[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Strings &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
EVENTS = [
    "Heater on the sensor mast checked and working.",
    "Battery replaced after a low voltage warning.",
    "Snow cleared from the rain gauge.",
    "Sensor recalibrated against the reference thermometer.",
    "Ice on the anemometer, so the wind readings for the morning are unreliable.",
    "Annual service of the station completed.",
    "Fence repaired after a storm.",
    "Data logger restarted after a power cut, and no readings were lost.",
    "Heaters on the mast replaced.",
    "Visited twice to check the heater.",
]
ON_WARM_DAYS = {EVENTS[2]: "Grass cut around the rain gauge.", EVENTS[4]: "Anemometer bearings greased."}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


def logbook():
    """A technician's note for every station and day of 2025, written from that day's readings."""
    days = {}
    for station, hour, celsius in year_of_readings():
        days.setdefault((hour[:10], station), []).append(celsius)
    for (day, station), temperatures in sorted(days.items()):
        known = [celsius for celsius in temperatures if celsius is not None]
        if not known:
            yield station, day, "Data logger failed overnight, and there are no readings for the whole day."
            continue
        low, high = min(known), max(known)
        if high < 0:
            weather = f"Frost all day, between {low} and {high} degrees."
        elif low < 0:
            weather = f"Night frost down to {low} degrees, and a thaw to {high} by the afternoon."
        else:
            weather = f"Above freezing all day, between {low} and {high} degrees."
        day_of_year = datetime.strptime(day, "%Y-%m-%d").timetuple().tm_yday
        number = (day_of_year * 37 + list(STATIONS).index(station) * 101) % 23
        event = EVENTS[number] if number < len(EVENTS) else ""
        if low >= 0:
            event = ON_WARM_DAYS.get(event, event)
        yield station, day, f"{weather} {event}".strip()


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})
notes = pd.DataFrame(list(logbook()), columns=["station", "day", "note"])
typed = pd.Series([" Bergen", "BERGEN", "bergen ", "Oslo", " oslo", "Tromsø", "TROMSO", "Tromso ", "Svalbard"],
                  name="station")                                       # station names as people typed them

print("pandas", pd.__version__, "|", len(readings), "readings,", len(notes), "notes,", len(typed), "typed names")

pandas 2.2.3 | 35040 readings, 1460 notes, 9 typed names


**1.** Count the notes at each station that mention the anemometer, in any case.

In [2]:
anemometer = notes["note"].str.contains("anemometer", case=False)

anemometer.groupby(notes["station"]).sum()

station
Bergen      16
Oslo        15
Svalbard    15
Tromso      15
Name: note, dtype: int64

`case=False` finds both `Anemometer bearings greased.` and `Ice on the anemometer`. Grouping the `True` and `False` values by station and summing them counts the `True` ones.

**2.** Count the notes longer than 120 characters, with `.str.len()`.

In [3]:
print((notes["note"].str.len() > 120).sum())

96


`.str.len()` gives a length for every note, the comparison makes a mask, and `sum` counts it.

**3.** Make a Series of every note with `degrees` replaced by `°C`, and print the first two.

In [4]:
celsius_notes = notes["note"].str.replace("degrees", "°C")

print(celsius_notes.head(2).tolist())

['Night frost down to -3.7 °C, and a thaw to 2.8 by the afternoon.', 'Night frost down to -5.3 °C, and a thaw to 1.2 by the afternoon. Heater on the sensor mast checked and working.']


`str.replace` reads `degrees` as plain text, the default in pandas 2, and replaces it everywhere it appears.

**4.** Among the notes that start with `Frost all day`, find the coldest low, using `startswith` and `extract`.

In [5]:
frost = notes[notes["note"].str.startswith("Frost all day")]
lows = frost["note"].str.extract(r"(?P<low>-?\d+\.\d)")["low"].astype(float)

coldest = lows.idxmin()
print(lows.min(), "|", notes.loc[coldest, "station"], notes.loc[coldest, "day"])

-17.3 | Svalbard 2025-01-12


The first number in a frost-all-day note is the low, so a pattern with one group is enough. `idxmin` is the index label of the coldest low, and `notes.loc` reads that note's station and day.

**5.** Count the battery replacements in each month, using the month in the `day` column, which `str.split("-", expand=True)` or `str[5:7]` can take out.

In [6]:
month = notes["day"].str.split("-", expand=True)[1]
battery = notes["note"].str.contains("Battery")

battery.groupby(month).sum()

1
01    8
02    4
03    4
04    5
05    7
06    4
07    5
08    7
09    4
10    5
11    7
12    4
Name: note, dtype: int64

A pattern of one character, `-`, is read as plain text, so the split needs no `regex=False`. Column 1 of the result is the month, and grouping the mask by it counts each month's replacements. `notes["day"].str[5:7]` slices the same two characters out of every value.

---

&#8592; **Back to:** [Strings](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/17-strings.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
